# Vues par semaine — générique vs. personnalisées (Σ conventions collectives)

Playground interactif pour comparer, par contribution, les vues hebdomadaires de
la page **générique** et la **somme** des vues de toutes les pages **personnalisées**
(une par convention collective). Les URLs personnalisées sont raccordées de part et
d'autre de la migration de slugs (schéma plat `/contribution/{idcc}-{slug}` **et**
imbriqué `/contribution/{slug}/{idcc}-{cc}`), pour que chaque courbe reste continue.

**Comment tester :** édite la cellule *Paramètres* ci-dessous (dates, métrique,
`DEMO`, ou la liste `CONTRIBUTIONS`), puis relance les cellules (`Shift+Enter`).
Source live = API Matomo (lit `MATOMO_*` depuis `analysis/.env`).

In [ ]:
# Recharge automatiquement le code modifié du module (évite de devoir redémarrer
# le kernel à chaque édition de contrib_weekly_views.py).
%load_ext autoreload
%autoreload 2
%matplotlib inline

import pandas as pd
from datetime import date

from analysis.reports.contrib_weekly_views import (
    CONTRIBUTIONS,
    METRICS,
    aggregate_weeks,
    build_chart,
    build_demo_data,
    comparison_summary,
    decline_table,
    fetch_live,
    relative_comparison,
    weekly_comparison,
)

# Afficher TOUTES les lignes / colonnes des tableaux (aucune troncature "...").
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

# ── Paramètres — édite puis relance les cellules ────────────────────────────
START = "2024-09-01"
END = date.today().isoformat()
METRIC = "views"  # "views" (pageviews) | "visits"
DEMO = False  # True = données synthétiques instantanées (aucun appel Matomo)

metric_field, metric_label = METRICS[METRIC]
[(c.key, c.label, c.generic_slug) for c in CONTRIBUTIONS]

## 1. Récupération + agrégation

⏳ **Le 1er appel Matomo prend ~2-3 min** (~100 semaines) et affiche un message de
progression — la cellule n'est **pas** figée. Le résultat est **mis en cache** sur
disque : les exécutions suivantes (autre métrique, graphe, tableaux) sont instantanées.
Pour un aperçu immédiat sans Matomo, mets `DEMO = True` dans les paramètres.

In [ ]:
weeks = build_demo_data() if DEMO else fetch_live(START, END)
df = aggregate_weeks(weeks, metric_field)
print(f"{len(df)} semaines : {df['week'].iloc[0]} → {df['week'].iloc[-1]}")
df  # tableau brut complet — toutes les semaines s'affichent

## 2. Comparaison des baisses (pic → dernière semaine)

In [ ]:
decline_table(df)

### … et semaine par semaine (toutes les semaines)

Pour chaque contribution : `générique`, `perso Σ` (somme des CC), `total`, et la
variation hebdomadaire du total (`Δ%/sem`, négatif = baisse). Toutes les semaines
sont listées — utile pour comparer les baisses semaine après semaine.

In [ ]:
# Semaine par semaine : toutes les semaines s'affichent (max_rows réglé plus haut).
weekly_comparison(df)

## 2c. Congés vs retraite — % de différence

Pour juger si les changements sur **congés** sont pertinents, on compare sa
trajectoire à celle de **retraite** (contribution témoin, non modifiée de la même
façon). Les deux *totaux* (générique + perso Σ) sont indexés à **100 à la 1re
semaine** : l'`écart %` isole la performance *relative*, indépendamment du volume.

- `écart %` durablement **négatif** → congés perd de l'audience plus vite que
  retraite : le changement n'aide pas (ou dégrade).
- `écart %` **positif / stable** → congés tient aussi bien ou mieux : changement neutre/positif.

In [ ]:
# Synthèse : baisse totale de chaque contribution + écart de baisse entre les deux.
comparison_summary(df)

In [ ]:
# % de différence semaine par semaine (totaux indexés base 100 à la 1re semaine).
# écart % > 0 = congés fait mieux que retraite ; < 0 = congés fait moins bien.
relative_comparison(df)

## 3. Graphe

Panneau du haut : vues absolues par semaine. Panneau du bas : chaque série indexée
à 100 à son propre pic, pour comparer la *forme* des baisses indépendamment du volume.

In [ ]:
from pathlib import Path

from IPython.display import Image

out = Path("../output")
out.mkdir(parents=True, exist_ok=True)
png = out / "contrib_weekly_playground.png"
build_chart(df, metric_label, png)
Image(str(png))

## Aller plus loin

- **Changer de métrique :** `METRIC = "visits"` dans les paramètres.
- **Suivre une autre contribution :** ajoute une entrée à `CONTRIBUTIONS` (voir
  `analysis/src/analysis/reports/contrib_weekly_views.py`) avec sa `generic_slug`
  et une couleur, puis relance. Le matching plat + imbriqué s'applique automatiquement.
- **Exporter :** `df.to_csv("../output/mon_export.csv", index=False)`.
- **Inspecter une semaine brute :** `weeks[list(weeks)[-1]][:5]`.